In [1]:
import os, gzip, pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [3]:
# paths

notebook_dir = os.getcwd()

os.chdir(notebook_dir)

In [12]:
# annealing schedule / parameter conversion
import pandas as pd

df = pd.read_csv('../QSim_Data/09-1265A-E_Advantage_system5_4_annealing_schedule.csv', sep=',', decimal='.')
sa = np.asarray(df['s'])
Ga = np.asarray(df['A(s) (GHz)']) * np.pi
Ja = np.asarray(df['B(s) (GHz)']) * np.pi

def _idx(E, gc=1.0):
    return np.argmin(np.abs(Ga - gc * E * Ja))

def Jc(E, gc=1.0):
    return E * Ja[_idx(E, gc)]

def Gc(E, gc=1.0):
    return Ga[_idx(E, gc)]

def Jcp(E, gc=1.0):
    return np.gradient(E * Ja, sa)[_idx(E, gc)]

def Gcp(E, gc=1.0):
    return np.gradient(Ga, sa)[_idx(E, gc)]

def tQ(ta_ns, E, gc=1.0):
    return ta_ns / gc * Jc(E, gc) / (Jcp(E, gc) / Jc(E, gc) - Gcp(E, gc) / Gc(E, gc))

def hcoJc(hz, E, gc=1.0):
    return hz

def en_to_lambda(E, gc=1.0):
    num = (Jcp(E, gc) / Jc(E, gc) - Gcp(E, gc) / Gc(E, gc)) / Jc(E, gc)
    den = (Jcp(1.0, gc) / Jc(1.0, gc) - Gcp(1.0, gc) / Gc(1.0, gc)) / Jc(1.0, gc)
    return num / den

In [13]:
# Load Q Sim data
try:
    with open('../QSim_Data/Fig_6_QSim_data.pkl', 'rb') as f:
        ta_vals = pickle.load(f)

except:
    print("Something went wrong")

In [14]:
h_targ = -0.657933 # -0.081113 # 

# Plot aesthetics
ms = 7.5    # Marker size
fs = 21     # Font size
lw = 1.0    # Line width

# Set Global Matplotlib params
plt.switch_backend('pgf')
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": "Times New Roman",
    "font.size": fs,
})

fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(10.0, 8.0), sharex=True, sharey=True)

for key, ta_list in ta_vals.items():
    x_temp = np.ones(len(ta_vals[(h_targ, key[1])])) * key[1]
    y_temp = np.array(ta_list)

    for axi in range(2):
        if axi == 0:
            mask = (y_temp < 1.0)
            axs[axi].plot(x_temp[mask], y_temp[mask], color='C0', marker='o', ms=ms, ls='')
            axs[axi].set_ylabel(r'annealing time $t_a~[\mathrm{\mu s}]$', fontsize=fs)
            axs[axi].set_yscale('log')
            
        elif axi == 1:
            mask = (y_temp < 1.0) & (5e-1 <= tQ(1000 * y_temp, key[1])) & (tQ(1000 * y_temp, key[1]) <= 5e0)
            axs[axi].plot(x_temp[mask], y_temp[mask], color='C0', marker='o', ms=ms, ls='')
            axs[axi].set_ylabel(r'annealing time $t_a~[\mathrm{\mu s}]$', fontsize=fs)
            axs[axi].set_yscale('log')
            axs[axi].set_xlabel(r'scaling factor $E$', fontsize=fs)

axs[0].text(-0.085, 1.0, r'(a)', transform=axs[0].transAxes, 
                fontsize=fs, fontweight='bold', va='top', ha='right')
axs[1].text(-0.085, 1.0, r'(b)', transform=axs[1].transAxes, 
                fontsize=fs, fontweight='bold', va='top', ha='right')
plt.subplots_adjust(hspace=0.05, bottom=0.0)

out_filename = 'Fig_6.pdf'
plt.savefig(out_filename, dpi=300, bbox_inches='tight')
plt.close()